In [11]:
import gwpopulation
gwpopulation.set_backend('jax')
xp = gwpopulation.utils.xp
import astropy
Planck15 = astropy.cosmology.Planck15
z_at_value = astropy.cosmology.z_at_value
import numpy as np
import jax.numpy as jnp

import jax
import jax_tqdm

import os 

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NPOC'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.25'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'
os.environ['NPROC'] = '4'
os.environ['intra_op_parallelism_threads'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def selection_function(weights, total_samples):
    return jnp.sum(weights) / total_samples

def selection_function_log_covariance(weights_n, weights_m, total_samples):
    assert weights_n.shape == weights_m.shape
    mu_n, mu_m = selection_function(weights_n, total_samples), selection_function(weights_m, total_samples)
    cov = jnp.sum(weights_n * weights_m) / total_samples / mu_n / mu_m - 1
    return cov / (total_samples-1)

def likelihood_log_correction(weights, total_samples, Nobs):
    var = selection_function_log_covariance(weights, weights, total_samples)
    return Nobs * (Nobs+1) * var / 2

def reweighted_event_bayes_factors(event_pe_weights):
    # Nobs, NPE = event_pe_weights.shape
    return jnp.mean(event_pe_weights, axis=1)

def event_log_covariances(event_pe_weights_n, event_pe_weights_m):
    assert event_pe_weights_m.shape == event_pe_weights_n.shape
    Nobs, NPE = event_pe_weights_n.shape

    mu_n = reweighted_event_bayes_factors(event_pe_weights_n)
    mu_m = reweighted_event_bayes_factors(event_pe_weights_m)

    cov = np.mean(event_pe_weights_n*event_pe_weights_m, axis=1) / mu_n / mu_m - 1
    return cov / (NPE - 1)

def log_likelihood_covariance(vt_weights_n, vt_weights_m, event_pe_weights_n, event_pe_weights_m, total_samples):
    Nobs, NPE = event_pe_weights_n.shape

    event_covs = event_log_covariances(event_pe_weights_n, event_pe_weights_m)
    vt_cov = selection_function_log_covariance(vt_weights_n, vt_weights_m, total_samples)

    return jnp.sum(event_covs) + Nobs**2 * vt_cov

def error_statistics(vt_weights, event_weights, total_samples, include_likelihood_correction=True):
    '''
    vt_weights: jnp.ndarray
        array of shape [n_samples, n_injections] where the 0th axis indexes the hyperposterior
        sample, and the 1st axis indexes the injections

    event_weights: jnp.ndarray
        array of shape [n_samples, n_obs, n_pe] where the 0th axis indexes the hyperposterior
        sample, the first axis is the event number and the last axis is the single event sample

    total_samples: float
        total number of injections for vt_weights

    returns:
    tuple of precision statistic, accuracy statistic and error statistic
    '''
    
    # compute weights according to Eq. 39
    length, Nobs, NPE = event_weights.shape
    axis = jnp.arange(length)
    arr_n, arr_m = jnp.meshgrid(axis, axis, indexing='ij')
    f = lambda n, m: log_likelihood_covariance(vt_weights[n], vt_weights[m], event_weights[n], event_weights[m], total_samples)
    _f = lambda x: f(x, x)
    variances = jax.lax.map(_f, axis)

    @jax_tqdm.scan_tqdm(length, print_rate=1, tqdm_type='std')
    def weight_func(carry, n):
        _f = lambda x: f(arr_n[n,x], arr_m[n,x])
        meanw = jnp.mean(jax.lax.map(_f, axis), axis=0)
        if include_likelihood_correction:
            meanw = likelihood_log_correction(vt_weights[n], total_samples, Nobs) - meanw
        return carry, meanw

    weight_func = jax_tqdm.scan_tqdm(length, print_rate=1, tqdm_type='std')(weight_func)
    _, weights = jax.lax.scan(weight_func, 0., xs=axis)

    precision = (jnp.mean(variances) - jnp.mean(weights)) / 2 / jnp.log(2)
    accuracy = jnp.var(weights) / 2 / jnp.log(2)
    error = precision + accuracy
    
    return precision, accuracy, error 


In [ ]:
vt_weights = jnp.array(np.random.uniform(size=(1000, 100_000)))
event_weights = jnp.array(np.random.uniform(size=(1000, 100, 1000)))
total_samples = 5e6

error_statistics(vt_weights, event_weights, total_samples, include_likelihood_correction=True)

Running for 1,000 iterations: 100%|██████████| 1000/1000 [00:46<00:00, 21.64it/s]


(Array(0.14170836, dtype=float64),
 Array(4.31101021e-09, dtype=float64),
 Array(0.14170836, dtype=float64))